In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_non_real_estate_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "NonRealEstateId string, SourceSystem string,  PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, NonRealEstateKey string, BatchId string,"
        #     "NonRealEstateOldKey int, NonRealEstateCode string,"
        #     "NonRealEstateComment string, NonRealEstateTypeCode string, NonRealEstateDescription string,  NonRealEstateAmount decimal(19,2),"
        #     "NonRealEstateCurrency string, NonRealEstateItemCount double, NonRealEstateItemDescription string,"
        #     "CollateralKey int,"
        #     "OwnerPartyKey int, NonRealEstateAppliedDate date,"
        #     "CashAccountNumber string, CashAccountTypeCode string,"
        #     "CashAccountType string, AccountBalanceFreeze string,"
        #     "CashAccountBank string, LiquidationAmount decimal(19,2),"
        #     "TimetoLiquidate string, ISIN string,"
        #     "FaceValue decimal(19,2),"
        #     "ServiceDate timestamp, CarModel string,"
        #     "CarFirstCircDate string, CarTypeCode string,"
        #     "CarType string, InstrumentIssuerKey string,"
        #     "InsurancePolicyNo string, InsurancePolicyIssuer string,"
        #     "InsuranceExpiredDate timestamp, InsuranceTypeCode string,"
        #     "InsuranceType string, InsuranceStartDate timestamp,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.NonRealEstate"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "thesis",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "portfolio_id_thesis",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== prcollateraldet ==================

        prcollateraldet_extra_filters = None

        if prcollateraldet_extra_filters is not None:
            prcollateraldet_extra_filters.append(date_filter)
            prcollateraldet_final_filters = " AND ".join(prcollateraldet_extra_filters)
        else:
            prcollateraldet_final_filters = date_filter

        prcollateraldet_selected_fields = [
            "id",
            "previous_id",
            "nre_collateral_code",
            "comments",
            "type_code",
            "collateral_amount",
            "currency_code",
            "item_count",
            "item_descr",
            "collateral_id",
            "person_id",
            "date_applied",
            "account_number",
            "account_type",
            "account_balance_freeze",
            "account_bank",
            "liquidation_amount",
            "time_to_liquidate",
            "ISIN",
            "face_value",
            "service_date",
            "car_model",
            "car_first_circ_date",
            "car_type",
            "instrum_issuer",
            "insur_policy_no",
            "insur_policy_issuer",
            "insur_expiry_date",
            "insur_type_code",
            "descr",
            "insur_start_date",
            "reference_key",
        ]

        prcollateraldet = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prcollateraldet",
            prcollateraldet_final_filters,
            prcollateraldet_selected_fields,
        )

        # ================== prcollateral ==================

        prcollateral_extra_filters = None

        if prcollateral_extra_filters is not None:
            prcollateral_extra_filters.append(date_filter)
            prcollateral_final_filters = " AND ".join(prcollateral_extra_filters)
        else:
            prcollateral_final_filters = date_filter

        prcollateral_selected_fields = [
            "id",
            "portfolio_owner",
            "businessdate",
        ]

        prcollateral = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prcollateral",
            prcollateral_final_filters,
            prcollateral_selected_fields,
        )

        # Join Conditions
        Portfolio_Prcollateral_join_condition = (
            portfolio["portfolio_id_thesis"] == prcollateral["portfolio_owner"]
        )

        Prcollateral_Prcollateraldet_join_condition = (
            prcollateral["id"] == prcollateraldet["collateral_id"]
        )

        # joins
        target_table = prcollateraldet.join(
            prcollateral,
            Prcollateral_Prcollateraldet_join_condition,
            "left",
        ).join(
            portfolio,
            Portfolio_Prcollateral_join_condition,
            "left",
        )

        primary_key_fields = [
            prcollateraldet.id,
            lit("thesis"),
        ]

        target_table = target_table.withColumn(
            "NonRealEstateId",
            sha2(concat_ws(";", *primary_key_fields), 256),
        )

        result_df = target_table.select(
            target_table.NonRealEstateId,
            lit("thesis").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            portfolio.portfolio_id_thesis.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            prcollateraldet.id.cast("string").alias("NonRealEstateKey"),
            target_table.previous_id.alias("NonRealEstateOldKey"),
            target_table.nre_collateral_code.alias("NonRealEstateCode"),
            target_table.comments.alias("NonRealEstateComment"),
            target_table.type_code.alias("NonRealEstateTypeCode"),
            target_table.descr.cast("string").alias("NonRealEstateDescription"),
            target_table.collateral_amount.alias("NonRealEstateAmount"),
            target_table.currency_code.alias("NonRealEstateCurrency"),
            target_table.item_count.alias("NonRealEstateItemCount"),
            target_table.item_descr.alias("NonRealEstateItemDescription"),
            target_table.collateral_id.alias("CollateralKey"),
            target_table.person_id.alias("OwnerPartyKey"),
            target_table.date_applied.alias("NonRealEstateAppliedDate"),
            target_table.account_number.alias("CashAccountNumber"),
            target_table.account_type.alias("CashAccountTypeCode"),
            target_table.account_type.alias("CashAccountType"),
            target_table.account_balance_freeze.alias("AccountBalanceFreeze"),
            target_table.account_bank.alias("CashAccountBank"),
            target_table.liquidation_amount.alias("LiquidationAmount"),
            target_table.time_to_liquidate.alias("TimetoLiquidate"),
            target_table.ISIN.alias("ISIN"),
            target_table.face_value.alias("FaceValue"),
            target_table.service_date.alias("ServiceDate"),
            target_table.car_model.alias("CarModel"),
            target_table.car_first_circ_date.alias("CarFirstCircDate"),
            target_table.car_type.alias("CarTypeCode"),
            target_table.car_type.alias("CarType"),
            target_table.instrum_issuer.alias("InstrumentIssuerKey"),
            target_table.insur_policy_no.alias("InsurancePolicyNo"),
            target_table.insur_policy_issuer.alias("InsurancePolicyIssuer"),
            target_table.insur_expiry_date.alias("InsuranceExpiredDate"),
            target_table.insur_type_code.alias("InsuranceTypeCode"),
            target_table.insur_type_code.alias("InsuranceType"),
            target_table.insur_start_date.alias("InsuranceStartDate"),
            prcollateral.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================
        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]
        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_non_real_estate_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)